# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIRˆ<sup>2</sup>](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. We reference all dataset elements (record sets, fields, columns) strictly by their `@id` values for reproducibility and clarity.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)
# Obtain metadata (as a dict)
metadata = dataset.metadata.to_json()

print(f"\nTitle: {metadata.get('name', '')}\n")
print(f"Description: {metadata.get('description', '')}\n")
print(f"Published: {metadata.get('datePublished', '')}")
print(f"License: {metadata.get('license', '')}")
print(f"Cite as: {metadata.get('citeAs', '')}")

## 2. Data Overview
Review available record sets, their `@id`, and show all field/column (attribute) `@id` for each.

We use `metadata['recordSet']` and reference all children by their `@id`. In mlcroissant, the term 'field' usually refers to a piece of data (such as column in a table).

In [ ]:
# Helper: load all record sets

record_sets = []
key = 'recordSet'
if key in metadata:
    record_sets = metadata[key]
    print(f"Discovered {len(record_sets)} record sets:")
    for i, rs in enumerate(record_sets):
        # Ensure each is a dict with '@id' and possibly other properties
        rs_id = rs.get('@id', '<none>') if isinstance(rs, dict) else rs
        print(f"  {i}. Record set @id: {rs_id}")
        # Print available fields/columns - usually under 'field' or 'column'
        if isinstance(rs, dict):
            if 'field' in rs:
                fields = rs['field']
                if isinstance(fields, list):
                    print('   Fields:')
                    for f in fields:
                        print(f"    - {f.get('@id', f)}")
            elif 'column' in rs:
                columns = rs['column']
                if isinstance(columns, list):
                    print('   Columns:')
                    for col in columns:
                        print(f"    - {col.get('@id', col)}")
else:
    print("No record sets found in dataset metadata.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

First, we extract all data from the discovered record sets into pandas DataFrames, referencing record set `@id` as the key.

In [ ]:
# We extract ALL record sets; this will autoselect appropriate internal Croissant @id

dataframes = {}
record_set_ids = []

# Parse the record sets for their @id
for rs in record_sets:
    if isinstance(rs, dict) and '@id' in rs:
        record_set_ids.append(rs['@id'])
    elif isinstance(rs, str):
        record_set_ids.append(rs)
# Print which we are loading
print(f"RecordSet @ids to load: {record_set_ids}")

# Load all discovered record sets
for record_set_id in record_set_ids:
    try:
        print(f"Loading records for record set: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        # If the records exist and are dicts, load as DataFrame
        if records and isinstance(records[0], dict):
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"  Columns: {list(dataframes[record_set_id].columns)}")
        else:
            print(f"  No tabular data for {record_set_id} or not dict rows.")
    except Exception as e:
        print(f"  Failed to load records for {record_set_id}", e)

# Choose the main record set for downstream analysis (the one with the most columns/rows)
main_record_set_id = None
if dataframes:
    main_record_set_id = max(dataframes, key=lambda x: dataframes[x].shape[1])
    print(f"\nChosen main record set: {main_record_set_id}")
    print(f"Available columns:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No data loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. You may select numeric or categorical fields by their `@id` (column name) as discovered above.

As an example, we will filter for proteins with a high molecular weight (`MW`), normalize the `MW` field, and group by protein description or accession if it exists.

In [ ]:
# Choose a numeric field. Look for a column ending in MW (molecular weight), 'Abundance', 'Peptides', etc.
# For this example, we auto-pick an appropriate numeric field from columns

import numpy as np

df = dataframes[main_record_set_id]

# Find likely numeric columns
numeric_candidates = [c for c in df.columns if df[c].dtype in [np.float64, np.int64, float, int] or pd.api.types.is_numeric_dtype(df[c])]
if not numeric_candidates:
    # try to coerce numerics
    for c in df.columns:
        try:
            df[c] = pd.to_numeric(df[c])
            if pd.api.types.is_numeric_dtype(df[c]):
                numeric_candidates.append(c)
        except Exception:
            pass

print(f"Candidate numeric fields: {numeric_candidates}")
numeric_field_id = None
if numeric_candidates:
    # Prioritize MW or abundance
    for pref in ['MW', 'Abundance', 'PEPTIDES', 'Coverage']:
        matched = [c for c in numeric_candidates if pref.lower() in c.lower()]
        if matched:
            numeric_field_id = matched[0]
            break
    else:
        numeric_field_id = numeric_candidates[0]

print(f"Chosen numeric field: {numeric_field_id}")
# Now filter records where MW > threshold
threshold = 50000  # e.g., high molecular weight
if numeric_field_id:
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered rows with {numeric_field_id} > {threshold}: {len(filtered_df)} proteins")
    display(filtered_df.head())
    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} (z-score):")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Try grouping by a categorical field
    group_field_candidates = [c for c in df.columns if df[c].nunique() < len(df)//2 and not pd.api.types.is_numeric_dtype(df[c])]
    group_field = None
    for pref in ['Description', 'description', 'Accession', 'accession', 'Protein']:
        candidates = [c for c in group_field_candidates if pref in c]
        if candidates:
            group_field = candidates[0]
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().sort_values(ascending=False)
        print(f"\nGrouped mean {numeric_field_id} by {group_field}:")
        display(grouped_df.head())
else:
    print("No suitable numeric field found.")

## 5. Visualization
Visualize the distribution of the chosen numeric field (e.g., MW), and its normalized values. If possible, plot correlation to another numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=40)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    # Normalized
    if f"{numeric_field_id}_normalized" in filtered_df:
        plt.figure(figsize=(8,4))
        sns.histplot(filtered_df[f"{numeric_field_id}_normalized"].dropna(), kde=True)
        plt.title(f"Normalized {numeric_field_id} (z-score)")
        plt.xlabel(f"{numeric_field_id}_normalized")
        plt.show()
    # Pairwise numeric plot
    if len(numeric_candidates) > 1:
        pair_candidates = numeric_candidates[:3]
        sns.pairplot(df[pair_candidates].dropna())
        plt.suptitle("Pairwise relationships of numeric fields", y=1.02)
else:
    print("No numeric field for visualization.")

## 6. Conclusion
In this notebook, we demonstrated loading and exploring a Croissant-formatted proteomics dataset using mlcroissant. We referenced all entities (record sets, fields, columns) by their `@id`, filtered and visualized proteins by molecular weight, and normalized numeric fields for analysis. The FAIR^2 dataset structure makes it robust for reproducible and automated data workflows. Further study might focus on cross-sample comparative analyses, peptide mapping, and linking protein measurements to biological pathways.